# Benchmark Models With The CLI

This notebook creates a small local JSONL dataset, runs a small benchmark with
`python -m unified_stpp bench`, and inspects the resulting output directory.

It does not require Hugging Face data or a GPU. When opened directly in
Colab, the install cell clones the public repository and installs Seahorse.


## Install/import

The setup cell is safe to run locally or in a fresh Colab runtime. In Colab,
it clones the public repository if `unified_stpp` is not already importable.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/YahyaAalaila/STPPGC.git"
REPO_DIR = Path("/content/STPPGC")

if Path.cwd().name == "notebooks" and Path.cwd().parent.parent.joinpath("pyproject.toml").exists():
    os.chdir(Path.cwd().parent.parent)
elif Path.cwd().parent.joinpath("pyproject.toml").exists():
    os.chdir(Path.cwd().parent)

if importlib.util.find_spec("unified_stpp") is None:
    if not Path("pyproject.toml").exists() and Path("/content").exists():
        if not REPO_DIR.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
        os.chdir(REPO_DIR)

    if not Path("pyproject.toml").exists():
        raise RuntimeError("Run from the Seahorse repository root, or open this notebook in Colab with network access.")

    %pip install -q -e .

import json
import shutil
import subprocess
from pathlib import Path

import pandas as pd


## Generate small local JSONL data


In [ ]:
import json
from pathlib import Path

import numpy as np


def make_demo_sequences(n_sequences: int, *, seed: int, n_events: int = 10):
    rng = np.random.default_rng(seed)
    records = []
    for seq_id in range(n_sequences):
        gaps = rng.uniform(0.08, 0.22, size=n_events)
        times = np.cumsum(gaps)
        phase = np.linspace(0.0, 1.0, n_events) + 0.11 * seq_id
        locations = np.column_stack(
            [
                0.5 + 0.25 * np.sin(2 * np.pi * phase) + rng.normal(0, 0.01, n_events),
                0.5 + 0.25 * np.cos(2 * np.pi * phase) + rng.normal(0, 0.01, n_events),
            ]
        )
        records.append(
            {
                "times": np.round(times, 4).tolist(),
                "locations": np.round(np.clip(locations, 0.05, 0.95), 4).tolist(),
            }
        )
    return records


def write_jsonl(path: Path, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        for record in records:
            f.write(json.dumps(record) + "\n")

DATA_DIR = Path("runs/tutorials/02_benchmark_data")
write_jsonl(DATA_DIR / "train.jsonl", make_demo_sequences(4, seed=11))
write_jsonl(DATA_DIR / "val.jsonl", make_demo_sequences(2, seed=12))
write_jsonl(DATA_DIR / "test.jsonl", make_demo_sequences(2, seed=13))

print(DATA_DIR)
print("files:", sorted(p.name for p in DATA_DIR.iterdir()))


## Run a small benchmark

The overrides make `auto_stpp` and `deep_stpp` compact enough for a quick CPU
walkthrough. Remove or relax these overrides for real experiments.


In [ ]:
OUT_DIR = Path("runs/tutorials/02_small_benchmark")
shutil.rmtree(OUT_DIR, ignore_errors=True)

cmd = [
    sys.executable,
    "-m",
    "unified_stpp",
    "bench",
    "--presets",
    "poisson_gmm",
    "hawkes_gmm",
    "auto_stpp",
    "deep_stpp",
    "--dataset",
    str(DATA_DIR),
    "--seeds",
    "1",
    "--out",
    str(OUT_DIR),
    "--n_workers",
    "1",
    "--override",
    "training.n_epochs=1",
    "training.batch_size=2",
    "data.num_workers=0",
    "model.hidden_dim=8",
    "model.decoder.lookback=3",
    "model.decoder.max_history=3",
    "model.decoder.seq_len=3",
    "model.decoder.num_points=3",
    "model.decoder.n_prodnet=1",
    "model.decoder.hidden_size=8",
    "model.decoder.num_layers=1",
    "model.decoder.n_layers=1",
    "model.encoder.num_heads=1",
    "model.encoder.num_layers=1",
    "model.updater.num_heads=1",
    "data.adapter_kwargs.training_view=sliding_window",
    "data.adapter_kwargs.lookback=3",
    "data.adapter_kwargs.lookahead=1",
]

result = subprocess.run(cmd, text=True, capture_output=True, check=True)
print("Benchmark command completed.")
print(result.stdout.splitlines()[-1])


## Inspect output directory


In [ ]:
print("Output directory:", OUT_DIR)
print("Top-level files:")
for path in sorted(OUT_DIR.iterdir()):
    print("-", path.name)

with (OUT_DIR / "cell_index.json").open() as f:
    cell_index = json.load(f)

print("benchmark cells:", len(cell_index))
print(cell_index[0])


## Show resulting tables/metrics

Benchmark tables are present when the corresponding metric is available for the
run set.


In [ ]:
for table_name in ["table_test_nll_all.csv", "table_test_nll_exact.csv"]:
    table_path = OUT_DIR / table_name
    if table_path.exists():
        print(table_name)
        display(pd.read_csv(table_path))
    else:
        print(table_name, "not written for this benchmark")


## Inspect one saved run

Each benchmark cell points to a normal saved run directory. Those run directories
can be passed to `python -m unified_stpp evaluate ...`.


In [ ]:
first_run = Path(cell_index[0]["run_dir"])
print(first_run)
print(sorted(p.name for p in first_run.iterdir()))

run_result = first_run / "run_result.json"
if run_result.exists():
    print(json.dumps(json.loads(run_result.read_text()), indent=2)[:1000])


## What to try next

- Add more seeds with `--seeds 1 2 3`.
- Add more presets after checking the model capability matrix.
- Use the evaluation notebook/page to compute additional metrics for one saved
  run from `cell_index.json`.
